In [1]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()

openai_client = OpenAI(
    base_url=os.getenv("OPENAI_API_URL"),
    api_key=os.getenv("OPENAI_API_KEY")
)

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [4]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally, follow these steps:

1. **Install Ollama**:
   - Visit [https://ollama.com/download](https://ollama.com/download) and choose your operating system:
     - **macOS**: Download the `.pkg` and install it.
     - **Windows**: Download the `.msi` and install it.
     - **Linux**: Open a terminal and run:
       ```bash
       curl -fsSL https://ollama.com/install.sh | sh
       ```

2. **Run the Model**:
   - After installation, open a terminal and type:
     ```bash
     ollama run llama3
     ```
   - This will download the LLaMA 3 model (~4GB), start the model locally, and open a chat-like interface for interaction.

3. **Test the Local Server**:
   - To ensure it's running, execute:
     ```bash
     curl http://localhost:11434
     ```
   - You should receive a response similar to:
     ```json
     {"models": [...]}  
     ```

4. **Install the Python Client**:
   - Run the following command to install the client:
     ```bash
     pip install ollama
     ```

5

In [5]:
answer = assistant.rag('How do I run Olama locally?')
print(answer)

To run Olama locally, you need to set up your environment by installing Python, `uv`, Jupyter, Docker, and any other necessary tools for the module. While Codespaces provides a convenient starting environment, running the course locally is possible if you are comfortable with setting it up yourself. Make sure to document your setup and ensure that your environment is reproducible.


In [6]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
)

response.output_text

"To provide you with the best answer, I need a bit more context about the course you're referring to. If it's an online course, many have ongoing enrollments, while others may have specific start dates. If it's an in-person course or program, it may depend on availability. Check the course website or contact the course administrator for specific enrollment details."

In [7]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [8]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [9]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool]
)

In [10]:
len(response.output)

1

In [11]:
call = response.output[0]

In [12]:
call


ResponseFunctionToolCall(arguments='{"query":"course enrollment"}', call_id='call_CsPF9pQTGxh5wUdD2L3qWXHU', name='search', type='function_call', id='fc_tmp_i0vr2tqq3p', namespace=None, status='completed')

In [13]:
import json

args = json.loads(call.arguments)
args

{'query': 'course enrollment'}

In [14]:
call.name

'search'

In [15]:
results = search(**args)

In [16]:
result_json = json.dumps(results, indent=2)

In [17]:
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json,
}

In [18]:
messages.append(call)

In [19]:
messages.append(function_call_output)

In [20]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"course enrollment"}', call_id='call_CsPF9pQTGxh5wUdD2L3qWXHU', name='search', type='function_call', id='fc_tmp_i0vr2tqq3p', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_CsPF9pQTGxh5wUdD2L3qWXHU',
  'output': '[\n  {\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "When will the course be offered next?",\n    "answer": "Summer 2027.",\n    "doc_id": "bd31146b0e"\n  },\n  {\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions.",\n    "doc_id": "74eb249bbf"\n  },\n  {\n    "course": "llm-zoomcamp",\n    "section": "General Course-Relate

In [21]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool]
)

In [22]:
print(response.output_text)

Yes, you can still join the course! However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [23]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(775, 33)

In [25]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    # Prices per 1M tokens (example pricing)
    INPUT_PRICE_PER_MILLION = 0.15   # $0.15 / 1M input tokens
    OUTPUT_PRICE_PER_MILLION = 0.60  # $0.60 / 1M output tokens

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost
    }


# Your tokens
result = calculate_gpt54mini_price(775, 33)

print("Total Cost: $", round(result["total_cost"], 8))


Total Cost: $ 0.00013605


In [26]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        'call_id': call.call_id,
        'output': result_json,
    }

In [27]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'


messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

In [28]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool]
)

In [29]:
messages.extend(response.output)

for item in response.output:
    if item.type == 'function_call':
        print('function_call:', item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type == 'message':
        print('ASSISTANT:')
        print(item.content[0].text)

function_call: search {"query":"Can I join the course?"}


In [30]:
messages

[{'role': 'developer',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"Can I join the course?"}', call_id='call_qNPkiXYHQdoqswqmWVAV1ql1', name='search', type='function_call', id='fc_tmp_wbpksjsg8v', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_qNPkiXYHQdoqswqmWVAV1ql1',
  'output': '[\n  {\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": 

In [32]:
messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

it = 1

while True:
    print(f'iteration #{it}...')
    has_function_calls = False

    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=messages,
        tools=[search_tool]
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == 'function_call':
            print('function_call:', item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == 'message':
            print('ASSISTANT:')
            print(item.content[0].text)
    
    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search {"query":"join course"}
iteration #2...
ASSISTANT:
Yes, you can join the course! If you're interested in receiving a certificate, make sure to submit your project while submissions are still being accepted. You can start learning and submitting homework even without formal registration, as it’s not required to begin.

If you have any further questions about the course or need guidance on how to get started, feel free to ask! Is there any other area you'd like to explore?


In [33]:
def agent_loop(instructions, question, model='gpt-4o-mini') -> str:
    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    it = 1

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                print('ASSISTANT:')
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break
    
    return last_answer

In [34]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [35]:
result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"join course enrollment registration"}
iteration #2...
ASSISTANT:
Yes, you can still join the course! However, if you wish to receive a certificate, it's important to submit your project while submissions are still being accepted. 

You don’t need to officially register to start learning; you can begin right away and submit homework while the submission form is open. Just remember that registration is mainly for gauging interest before the start date.

If you have any more questions or if there are other areas you want to explore regarding the course, feel free to ask!


In [36]:
result

"Yes, you can still join the course! However, if you wish to receive a certificate, it's important to submit your project while submissions are still being accepted. \n\nYou don’t need to officially register to start learning; you can begin right away and submit homework while the submission form is open. Just remember that registration is mainly for gauging interest before the start date.\n\nIf you have any more questions or if there are other areas you want to explore regarding the course, feel free to ask!"

In [37]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "what's queen gambit?"

result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
ASSISTANT:
It seems that "Queen Gambit" may not relate to the course content, as there were no relevant entries found in the FAQ database. If you have a specific question about the course or its logistics, feel free to ask! Are there other areas related to the course that you would like to explore?


In [38]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [39]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [40]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={'question': 3.0, 'section': 0.5},
        filter_dict={'course': 'llm-zoomcamp'}
    )

In [41]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [42]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [43]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [49]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-4o-mini', client=openai_client)
)

In [50]:
result = runner.loop(
    prompt='How do I run Olama locally?',
    callback=callback,
)

-> Response received


-> Response received


In [51]:
result.cost

CostInfo(input_cost=Decimal('0.00023715'), output_cost=Decimal('0.0002058'), total_cost=Decimal('0.00044295'))

In [52]:
result.all_messages

[EasyInputMessage(content="\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searchers. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.\n", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"run Olama locally"}', call_id='call_BU8g36HfR4jk4AB9pDcgn

In [53]:
result2 = runner.loop(
    prompt='How do I run a different model?',
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [ ]:
runner.run();